# Ingestão de Base auxiliar com PySpark
This notebook uses **PySpark** to ingest, inspect, clean, and explore `base_auxiliar_fiap.csv`.

## 1. Imports and start SparkSession


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName('Ingestao Base Auxiliar')
    .config('spark.driver.memory', '2g')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('WARN')
print(f'Spark version: {spark.version}')

Spark version: 4.1.1


## 2. Define Schema & Ingest CSV
Defining the schema explicitly avoids a full scan for type inference and makes ingestion faster and more reliable.

In [2]:
FILE_PATH = '../../input_data/base_auxiliar_fiap.csv'


schema = StructType([
    StructField('id_cnpj',       StringType(), nullable=True),
    StructField('cd_cnae_prin',      StringType(), nullable=True),
    StructField('uf', StringType(), nullable=True),
    StructField('sacado_indice_liquidez_1m',      StringType(), nullable=True),
    StructField('cedente_indice_liquidez_1m',   StringType(), nullable=True),
    StructField('score_materialidade_evolucao',    StringType(), nullable=True),
    StructField('media_atraso_dias',     DoubleType(), nullable=True),
    StructField('indicador_liquidez_quantitativo_3m',       DoubleType(), nullable=True),
    StructField('share_vl_inad_pag_bol_6_a_15d',      StringType(), nullable=True),
    StructField('score_quantidade_v2',    StringType(), nullable=True),
    StructField('score_materialidade_v2',    StringType(), nullable=True),
])

df_raw = (
    spark.read
    .option('header', 'true')
    .schema(schema)

    .csv(FILE_PATH)
)

print(f'Rows ingested: {df_raw.count():,}')
print(f'Columns      : {len(df_raw.columns)}')
df_raw.printSchema()

Rows ingested: 4,612
Columns      : 11
root
 |-- id_cnpj: string (nullable = true)
 |-- cd_cnae_prin: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- sacado_indice_liquidez_1m: string (nullable = true)
 |-- cedente_indice_liquidez_1m: string (nullable = true)
 |-- score_materialidade_evolucao: string (nullable = true)
 |-- media_atraso_dias: double (nullable = true)
 |-- indicador_liquidez_quantitativo_3m: double (nullable = true)
 |-- share_vl_inad_pag_bol_6_a_15d: string (nullable = true)
 |-- score_quantidade_v2: string (nullable = true)
 |-- score_materialidade_v2: string (nullable = true)



In [3]:
df_raw.describe().show()


+-------+--------------------+------------------+----+-------------------------+--------------------------+----------------------------+------------------+----------------------------------+-----------------------------+-------------------+----------------------+
|summary|             id_cnpj|      cd_cnae_prin|  uf|sacado_indice_liquidez_1m|cedente_indice_liquidez_1m|score_materialidade_evolucao| media_atraso_dias|indicador_liquidez_quantitativo_3m|share_vl_inad_pag_bol_6_a_15d|score_quantidade_v2|score_materialidade_v2|
+-------+--------------------+------------------+----+-------------------------+--------------------------+----------------------------+------------------+----------------------------------+-----------------------------+-------------------+----------------------+
|  count|                4612|              4610|4253|                     4593|                      2463|                        4609|              4607|                              4592|                  

In [4]:
BRONZE_PATH = '../../output_data/bronze/base_auxiliar'

df_raw = df_raw.withColumn('ingestion_timestamp', F.current_timestamp()) \
        .withColumn('partition_date', F.current_date())

df_raw.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').partitionBy('partition_date').save(BRONZE_PATH)
print(f'Bronze layer saved to {BRONZE_PATH}')

Bronze layer saved to ../../output_data/bronze/base_auxiliar


In [5]:
df_bronze = spark.read.format('delta').load(BRONZE_PATH)
print(f'Rows in bronze layer: {df_bronze.count():,}')
df_bronze.printSchema()

Rows in bronze layer: 4,612
root
 |-- id_cnpj: string (nullable = true)
 |-- cd_cnae_prin: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- sacado_indice_liquidez_1m: string (nullable = true)
 |-- cedente_indice_liquidez_1m: string (nullable = true)
 |-- score_materialidade_evolucao: string (nullable = true)
 |-- media_atraso_dias: double (nullable = true)
 |-- indicador_liquidez_quantitativo_3m: double (nullable = true)
 |-- share_vl_inad_pag_bol_6_a_15d: string (nullable = true)
 |-- score_quantidade_v2: string (nullable = true)
 |-- score_materialidade_v2: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- partition_date: date (nullable = true)



In [6]:
df_bronze.show(5, truncate=False)

+----------------------------------------------------------------+------------+---+-------------------------+--------------------------+----------------------------+-----------------+----------------------------------+-----------------------------+-------------------+----------------------+--------------------------+--------------+
|id_cnpj                                                         |cd_cnae_prin|uf |sacado_indice_liquidez_1m|cedente_indice_liquidez_1m|score_materialidade_evolucao|media_atraso_dias|indicador_liquidez_quantitativo_3m|share_vl_inad_pag_bol_6_a_15d|score_quantidade_v2|score_materialidade_v2|ingestion_timestamp       |partition_date|
+----------------------------------------------------------------+------------+---+-------------------------+--------------------------+----------------------------+-----------------+----------------------------------+-----------------------------+-------------------+----------------------+--------------------------+--------------